# 01. Chuẩn hoá dataset DET/REC

Chọn kernel `.venv` (Python 3.13). Luồng: đọc nguồn → rà nhãn/box → duyệt → crop/nắn REC → dựng v2 → kiểm tra.
Hiện tái tạo từ v2, nguồn REC và quyết định đã duyệt. Bản ảnh/nhãn Roboflow đầu tiên không còn đầy đủ;
không coi v2 là bản sao của toàn bộ dữ liệu Roboflow gốc. Xem `docs/dataset.md` về đầu vào và giới hạn phục hồi.


## 1. Thiết lập

In [ ]:
from pathlib import Path
import sys

WORKSPACE = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "vn-plate-dataset_v2").is_dir() and (path / "metadata/baseline.json").is_file()
)
sys.path.insert(0, str(WORKSPACE))
DATASET_ROOT = WORKSPACE / "vn-plate-dataset_v2"
print(WORKSPACE)

## 2. Đọc nguồn có nhãn sơ bộ

Nếu còn bộ nguồn bên ngoài workspace, đổi `INPUT_ROOT`. Các ảnh đã duyệt cách ly được bỏ theo provenance.

In [ ]:
from dataset_preparation.dataset import Dataset, read_samples
from dataset_preparation.normalization import read_source

INPUT_ROOT = DATASET_ROOT
source_dataset = read_source(WORKSPACE, INPUT_ROOT)
dataset = Dataset.read(DATASET_ROOT)
samples = read_samples(WORKSPACE)
print(f"Nguồn: {len(source_dataset.det)} DET, {len(source_dataset.rec)} REC")
print(f"Quyết định đã duyệt: {len(samples)} dòng")

## 3. Rà nhãn, chất lượng và trùng ảnh

Đề xuất bỏ dấu phân cách/viết hoa, giữ số 0 đầu. Sửa chữ đọc sai cần quyết định đối chiếu ảnh nguồn.
Ngưỡng mờ/tương phản chỉ tính trên train. Ảnh mờ và ứng viên gần giống được gắn cờ để xem, không tự loại.


In [ ]:
from dataset_preparation.normalization import audit_source

source_audit = audit_source(source_dataset)
print("Ảnh đã rà:", source_audit["images"])
print("Đề xuất chuẩn hoá chuỗi:", len(source_audit["text_proposals"]))
print("Lỗi nhãn/hình học đầu vào:", len(source_audit["findings"]))
print("Ảnh có cờ chất lượng để xem lại:", source_audit["quality_flagged_images"])
print("Nhóm trùng pixel khác split:", len(source_audit["exact_cross_split_pixels"]))
print("Nhóm cùng chuỗi biển khác split:", len(source_audit["full_plate_cross_split"]))
print("Ứng viên gần giống cần xem:", len(source_audit["near_cross_split_candidates"]))

## 4. Tạo/siết bounding box trên ảnh gốc

`propose_line()` trong `dataset_preparation/line_envelopes.py` dùng khung ban đầu để tìm nét của từng dòng,
ước lượng đỉnh/chân ký tự và bốn cạnh. Hai tầng được xử lý riêng để tránh lấy nhầm nét của tầng bên cạnh.
`supported` chỉ là kết quả chẩn đoán; mọi đề xuất vẫn cần duyệt vì có thể bỏ sót chữ mờ.


In [ ]:
from collections import Counter
from dataset_preparation.normalization import propose_boxes

box_proposals = propose_boxes(source_dataset, method="native")
print("Dòng đã đề xuất:", len(box_proposals))
print(dict(Counter(row["diagnostics"]["status"] for row in box_proposals)))
assert all(row["review_required"] for row in box_proposals)

## 5. Xem khung và crop trước/sau

Cam là khung đầu vào, xanh là đề xuất. Xem kỹ ký tự đầu/cuối, mép trái/phải và chỗ giáp hai tầng.
Đổi `OFFSET` và `COUNT` để duyệt các nhóm khác; không coi số thành phần nét khớp số ký tự là bằng chứng đủ chữ.


In [ ]:
from dataset_preparation.preview import proposal_grid

FOCUS_IMAGES = {
    "val/det/images/det_00043.jpg", "val/det/images/det_00068.jpg",
    "val/det/images/det_00090.jpg", "val/det/images/det_00098.jpg",
    "val/det/images/det_00132.jpg", "test/det/images/det_00088.jpg",
}
focus = [row for row in box_proposals if row["det_image"] in FOCUS_IMAGES]
OFFSET, COUNT = 0, 8
display(proposal_grid(source_dataset, focus, count=COUNT, offset=OFFSET))

## 6. Nhánh siết bốn cạnh bằng DB và nét chữ

Nhánh cũ `glyph_quad()` trong `dataset_preparation/text_bounds.py` dùng xác suất chữ để chọn nét, fit bốn cạnh,
rồi map về ảnh gốc. Bản đồ xác suất không sửa pixel RGB hoặc đoán nhãn.
Cài `requirements/dataset-bounds.txt`, đặt `USE_DB_SUPPORT = True` để chạy thêm nhánh này.


In [ ]:
USE_DB_SUPPORT = False
if USE_DB_SUPPORT:
    db_proposals = propose_boxes(source_dataset, method="db", image_names=FOCUS_IMAGES)
    display(proposal_grid(source_dataset, db_proposals, count=6))
else:
    print("Nhánh native đã chạy đủ; DB là nhánh hỗ trợ có thể bật thêm.")

## 7. Duyệt và chốt khung/nhãn

`review_boxes()` nhận từng quyết định: `keep`, `proposal`, hoặc `manual`; phải có lý do và đủ mọi dòng.
Những đề xuất mất nét, méo hoặc cắt chữ phải bỏ; có thể nhập lại bốn góc thủ công.

Cell dưới phát lại quyết định đã duyệt của bản v2, gồm nhãn đã sửa, không tự chấp nhận đề xuất mới.
Ở một lượt xử lý mới, cần xem ảnh và điền lại quyết định trước khi xuất dữ liệu.


In [ ]:
from dataset_preparation.normalization import review_boxes

approved_by_key = {(row["det_image"], row["object_index"]): row for row in samples}
decisions = {}
for row in box_proposals:
    key = row["det_image"], row["object_index"]
    final = approved_by_key[key]
    decisions[key] = {
        "action": "keep" if row["input_points"] == final["det_points"] else "manual",
        "points": final["det_points"], "text": final["text"],
        "reason": "Phát lại quyết định đối chiếu ảnh nguồn đã lưu của v2",
    }
reviewed_boxes = review_boxes(source_dataset, box_proposals, decisions)
assert len(reviewed_boxes) == len(samples)
print("Đã đối chiếu quyết định của", len(reviewed_boxes), "dòng; không ghi lại v2.")

## 8. Crop và nắn REC từ nguồn đã chọn

Sau khi chốt box, `rectify_crop()` nắn vùng chữ. Nhánh `dataset` dùng ảnh DET;
nhánh `sources` dùng REC gốc đã được đối chiếu (số mẫu theo metadata). Hai loại nguồn có hệ tọa độ riêng.
Đây là lý do cần lưu cả `det_points` và `crop_points`. Chi tiết hàm: `dataset_preparation/geometry.py` (`rectify_crop()`).


In [ ]:
import hashlib
import io
from PIL import Image
from IPython.display import display
from dataset_preparation.pipeline import load_baseline, render_rec

baseline = load_baseline(WORKSPACE)
sample = next(row for row in samples if row["rec_image"] == "val/rec/images/rec_000111.jpg")
data = render_rec(WORKSPACE, sample)
assert hashlib.sha256(data).hexdigest() == baseline["files"][sample["rec_image"]]
print(sample["rec_image"], "| Nhãn:", sample["text"])
with Image.open(io.BytesIO(data)) as image:
    display(image.copy())

## 9. Xuất lại đầy đủ v2 và đối chiếu

`build_v2()` tạo lại sáu file nhãn từ quyết định duyệt, copy ảnh DET nguồn,
tạo lại mọi REC từ nguồn và giữ review list. Không copy ảnh REC thành phẩm để giả lập tái tạo.
Cell xuất vào thư mục tạm và xoá sau khi đối chiếu, không tạo v3 hoặc ghi đè v2 đang train.
Mọi file đầu ra phải khớp SHA với bản v2 đã chốt; quyết định mới khác bản chốt sẽ bị chặn.


In [ ]:
from tempfile import TemporaryDirectory
from dataset_preparation.pipeline import build_v2

with TemporaryDirectory(prefix="plate-v2-check-") as directory:
    rebuilt = build_v2(
        WORKSPACE, Path(directory) / "vn-plate-dataset_v2",
        input_root=INPUT_ROOT, reviewed_boxes=reviewed_boxes,
    )
    print("File đã dựng:", rebuilt["files"])
    print("REC tạo lại từ nguồn:", rebuilt["rec_rebuilt"])
    print("Khớp toàn bộ v2 đã duyệt:", rebuilt["matches_approved_v2"])

## 10. Kiểm tra toàn bộ v2 hiện tại

Kiểm checksum, giải mã ảnh, nhãn, hình học, liên kết DET–REC, trùng giữa các split và tái tạo từng REC.

In [ ]:
from dataset_preparation.pipeline import validate

result = validate(WORKSPACE)
assert result["technical_ready"]
print("Dataset hợp lệ:", result["technical_ready"])
print("Ảnh đã đọc:", result["images_decoded"])
print("Crop tái tạo khớp:", result["rec_reproduced_exactly"])
print("Polygon các tầng giao nhau:", result["cross_row_polygon_overlaps"])
print("Ảnh trùng pixel giữa các split:", result["exact_cross_split_pixel_groups"])

## 11. Xem DET và REC

Đổi `SPLIT`, `N_DET`, `N_REC`, `SEED` để xem thêm. Mặc định hiển thị 12 biển một tầng, 12 biển hai tầng và 18 crop REC.
Khung lấy trực tiếp từ nhãn hiện tại, chuỗi chữ đặt ngoài ảnh.

In [ ]:
from dataset_preparation.preview import det_grid, rec_grid

SPLIT = "val"
N_DET = 12
N_REC = 18
SEED = 42

### DET một tầng

In [ ]:
display(det_grid(dataset, split=SPLIT, lines=1, count=N_DET, seed=SEED))

### DET hai tầng

In [ ]:
display(det_grid(dataset, split=SPLIT, lines=2, count=N_DET, seed=SEED))

### REC

In [ ]:
display(rec_grid(dataset, split=SPLIT, count=N_REC, seed=SEED))

## 12. Đầu vào train REC

REC trong v2 đã crop và nắn. Chỉ resize/pad giữ tỉ lệ; `48 × 320` là ví dụ, chỉnh theo model.
Normalize theo framework train đang dùng. Không nắn lại crop REC đã lưu.

In [ ]:
from dataset_preparation.rec_preprocessing import iter_rec_samples, resize_pad

image_path, text = next(iter_rec_samples(DATASET_ROOT, "train"))
with Image.open(image_path) as image:
    model_input = resize_pad(image, height=48, width=320)
print("Nhãn:", text, "| Kích thước:", model_input.size)
display(model_input)

## 13. Kiểm tra hoặc tạo ZIP

ZIP chỉ chứa cây v2 để đưa sang máy train. Đặt `REBUILD_ZIP = True` khi cần đóng gói lại.
Muốn tái tạo REC ra thư mục trống riêng, dùng `reproduce_rec(WORKSPACE, samples, baseline, destination=...)`.

In [ ]:
from dataset_preparation.pipeline import package_dataset, verify_archive

REBUILD_ZIP = False
if REBUILD_ZIP:
    archive = package_dataset(WORKSPACE)
else:
    archive = verify_archive(WORKSPACE, WORKSPACE / "vn-plate-dataset_v2.zip")
print("Số file trong ZIP:", archive["files"])
print("SHA256:", archive["sha256"])

## 14. Thống kê cuối

Ảnh nguồn mờ hoặc che khuất vẫn có giới hạn. Có 699 review cũ chưa map được và 20 liên kết thiếu bằng chứng pixel lịch sử.
Kiểm tra kỹ thuật không thay thế đánh giá CER/accuracy sau train; chưa chứng nhận mọi họ ảnh nguồn độc lập giữa các split.

In [ ]:
print(f"{'Split':<8} {'DET':>7} {'REC':>7}")
for split, counts in result["split_counts"].items():
    print(f"{split:<8} {counts['det']:>7} {counts['rec']:>7}")
print(f"{'Tổng':<8} {result['det_images']:>7} {result['rec_images']:>7}")
print("Tổng polygon DET:", result["det_objects"])